# **YouTube Tutorial AI Summarizer**

An AI-powered Python tool that extracts transcripts from YouTube tutorials and generates concise, structured summaries using LLMs (Gemini / GPT / Ollama).

> We use youtube_transcript_api which allows you to get the transcript text of any youtube video.












In [3]:
!pip install youtube-transcript-api


   -------------------- ------------------- 1/2 [youtube-transcript-api]
   -------------------- ------------------- 1/2 [youtube-transcript-api]
   ---------------------------------------- 2/2 [youtube-transcript-api]



In [2]:
#imports libraries
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.formatters import TextFormatter, SRTFormatter
import re
from openai import OpenAI
from dotenv import load_dotenv
#from google.colabg import userdata # dotenv equevilant for google colab
from IPython.display import Markdown, display, update_display
import requests
import ollama
import os

In [3]:
# models
model_gemini='gemini-3-flash-preview'
model_gpt = 'gpt-4o-mini'
model_llama = 'gemma4'


01069130719

In [4]:
#creating gemini client

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")


gemini=OpenAI(api_key=google_api_key,base_url=GEMINI_BASE_URL)
print("your gemini client is ready to use")

API key found and looks good so far!
your gemini client is ready to use


In [5]:
requests.get("http://localhost:11434").content

b'Ollama is running'

In [6]:
#creating Ollama client

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
print("your ollama client is ready to use")

your ollama client is ready to use


In [7]:
import pysrt

def read_srt_file(file_path):
    # Load the SRT file
    subs = pysrt.open(file_path)
    # Extract text and clean up newlines
    full_text = " ".join([sub.text for sub in subs])
    return full_text.replace('\n', ' ')

# CLEANED PATH: Use 'r' and ONLY single backslashes
path = r'C:\Users\gama\Downloads\subtitle\011_Day_1.SRT'

try:
    transcript = read_srt_file(path)
    print("Success! Transcript loaded.")
    print(transcript[:])
except FileNotFoundError:
    print("STILL NOT FOUND. Please check if '011DAY_1.SRT' is actually named that, or if it's '01. Day 1.srt' in your folder.")

Success! Transcript loaded.
And now an exercise for you. You've just hopefully gone in and run this for other web pages, and you've maybe tried tweaking it for a different business purpose. The exercise is now to come in here and see if you can make your own message to GPT from scratch. So first of all, write a simple system prompt. Now you should think about something completely different to do here, perhaps still related to to summarizing information. So as an example, you might want to summarize the contents of an email. Or perhaps you might want to take an email and come up with a nice subject, a possible subject for that email that might be an example of a business task. So in the system prompt here, describe the task that you want it to do in the user prompt. You'll want to put in here the contents of the email or the thing to be translated or whatever the is going to be the prompt with the instruction. Then you now have to do some typing, just like I did some typing. It's your t

In [7]:
ytt = YouTubeTranscriptApi()
formatter = TextFormatter() # --> Plain text
# formatter = SRTFormatter() # --> With timestamps



In [9]:
def get_video_id(url):
    """Extracts video ID from a YouTube URL."""
    regex = r"(?:v=|\/)([0-9A-Za-z_-]{11}).*"
    match = re.search(regex, url)
    if match:
        return match.group(1)
    raise ValueError("Invalid YouTube URL")


def get_transcript(url):
    video_id = get_video_id(url)
    fetched_transcript = ytt.fetch(video_id)
    # ^ defaults to English transcript, for other language use:
    # fetched = ytt.fetch(video_id, languages=['de', 'en'])
    transcript_text = formatter.format_transcript(fetched_transcript)
    return transcript_text

In [18]:
system_prompt="""Role: Technical Instructor & Documentation Specialist

You are an expert Technical Instructor. Your mission is to convert YouTube tutorial transcripts into clean, reproducible Standard Operating Procedures (SOPs) or "Cheat Sheets."

Your core objectives:

Identify the Prerequisites: List any tools, software, or materials mentioned at the start.

Step-by-Step Logic: Convert the creator’s verbal instructions into a numbered list of clear, actionable steps.

Highlight "Gotchas": Identify warnings, common mistakes, or troubleshooting tips mentioned by the creator.

Technical Accuracy: Preserve specific commands, code snippets, measurements, or settings exactly as stated.

Formatting Rules:

Use Numbered Lists for the main workflow.

Use Code Blocks for commands, formulas, or specific settings.

Use Bold for UI elements (e.g., "Click the Submit button").

Place any "Pro-Tips" in a blockquote.
"""

In [12]:
def get_user_prompt(url):
    prompt = f"The following is a transcript for a tutorial \n"
    prompt += "Carefully read through this content, analyse and summerize it as told, respond in Markdown."
    prompt += "\nTranscript: \n\n" 
    prompt += get_transcript(url)
    return prompt

In [ ]:

def summerize_video(url):
    user_prompt = get_user_prompt(url)
    stream = ollama.chat.completions.create(
        model=model_llama,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        stream = True,
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [19]:
summerize_video("https://www.youtube.com/watch?v=VyWAvY2CF9c")

Here's a structured summary and breakdown of the key concepts covered in the transcript, organized logically for easy understanding.

---

## Comprehensive Overview of Deep Learning Workflow & Best Practices

The transcript covers the entire pipeline of working with machine learning models, from data preparation to mitigation of overfitting and model improvement.

### 1. The Core Workflow (From Data to Model)

* **Training the Model:** The process of feeding data to the model to learn patterns. The speaker emphasizes the cyclical nature of refinement.
* **Model Evaluation/Testing:** Using unseen data to determine if the model generalizes well.
* **Troubleshooting Overfitting:** Identifying and fixing issues where the model performs well on training data but poorly on test data.

### 2. Detailed Techniques for Improving Performance

#### A. Data Augmentation (Pre-Processing Enhancement)
* **Purpose:** To artificially increase the size and variability of the training dataset, making the model more robust and preventing overfitting.
* **Concept:** Creating modified versions of existing data (e.g., slightly rotating, cropping, or brightening an image) so the model learns the essential features rather than memorizing artifacts of the original small dataset.

#### B. Addressing Overfitting (Model Regularization)
* **Dropout:** A key regularization technique where, during training, a random percentage of neurons are temporarily ignored (set to zero). This forces the network not to rely too heavily on any single neuron, making the learned features more robust.
* **Batch Normalization (Implied):** While not explicitly detailed, the context suggests using techniques that stabilize training (like normalization layers).

#### C. System Improvements
* **Hyperparameter Tuning:** The process of adjusting settings that are *not* learned during training (like learning rate, number of layers) to achieve the best performance.

### 3. Specific Technical Deep Dives

#### A. Optimization & Loss Functions
* **Loss Function:** Measures how wrong the model's predictions are. The goal of training is to **minimize** this value.
* **Optimization (e.g., Gradient Descent):** The mathematical mechanism used to adjust the model's weights and biases iteratively to reduce the loss function.

#### B. Regularization Techniques (General)
* **L1/L2 Regularization:** Penalizing weights based on their magnitude. This encourages the model to use the simplest possible combination of features, effectively preventing weights from becoming too large (which signals over-reliance on specific features).

### 4. Feature Engineering and Data Preparation

* **Normalization/Standardization (Scaling):** Ensuring that all input features (columns) contribute equally to the learning process. Features with naturally larger ranges must be scaled down so they don't disproportionately influence the optimization steps.
* **Feature Selection:** Choosing only the most relevant input variables to prevent the "curse of dimensionality" and simplify the model.

### 5. Practical Pitfalls and Best Practices Summary

| Concept | Goal | Technique Used |
| :--- | :--- | :--- |
| **Data Scarcity** | Increase data variety. | Data Augmentation, Acquiring More Data. |
| **Overfitting** | Prevent memorization of training noise. | Dropout, L1/L2 Regularization, Early Stopping. |
| **Feature Importance** | Simplify the model. | Feature Selection, Regularization. |
| **Algorithm Drift** | Ensure consistent results. | Cross-Validation (Testing on multiple subsets). |
| **Uneven Data** | Ensure equal contribution from all features. | Scaling/Normalization (Scaling). |

---
***In essence, the transcript guides the listener through the engineering discipline of building AI: gather diverse data $\rightarrow$ scale and clean it $\rightarrow$ build the model $\rightarrow$ train it using optimization $\rightarrow$ monitor for overfitting $\rightarrow$ apply regularization techniques $\rightarrow$ repeat and improve.***